In [0]:
# Cell 1 — Setup (thay thế toàn bộ)
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

import boto3
import pandas as pd
import io

AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BUCKET_NAME    = ""
REGION         = "eu-north-1"

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

# ── HÀM MỚI: đọc tất cả CSV trong các timestamp subfolders ──
def read_csv_from_s3(table_name):
    """
    Plan mới: S3 có nhiều timestamp subfolders
    raw/{table}/20250101_080000/data.csv
    raw/{table}/20250101_080500/data.csv
    → list tất cả files, đọc và union lại
    """
    prefix = f"raw/{table_name}/"
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)

    if 'Contents' not in response:
        raise FileNotFoundError(f"Không tìm thấy file nào tại s3://{BUCKET_NAME}/{prefix}")

    # Lọc chỉ lấy file data.csv
    csv_keys = [obj['Key'] for obj in response['Contents']
                if obj['Key'].endswith('/data.csv')]

    if not csv_keys:
        # Fallback: thử đọc file cũ format (nếu [D] chưa chạy service mới)
        fallback_key = f"raw/{table_name}/{table_name}.csv"
        try:
            obj = s3.get_object(Bucket=BUCKET_NAME, Key=fallback_key)
            content = obj['Body'].read().decode('utf-8-sig')
            pandas_df = pd.read_csv(io.StringIO(content))
            spark_df = spark.createDataFrame(pandas_df)
            print(f"   [FALLBACK] Dùng file cũ: {fallback_key}")
            return spark_df
        except:
            raise FileNotFoundError(
                f"Không tìm thấy data.csv nào tại {prefix}\n"
                f"-> [D] Khang cần chạy incremental_export.py trước"
            )

    # Đọc tất cả CSV files và union
    dfs = []
    for key in sorted(csv_keys):  # sort để newest cuối
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
        content = obj['Body'].read().decode('utf-8-sig')
        pdf = pd.read_csv(io.StringIO(content))
        dfs.append(spark.createDataFrame(pdf))
        print(f"   Đọc: {key} ({len(pdf)} rows)")

    # Union tất cả, dedup theo id
    from functools import reduce
    from pyspark.sql import DataFrame
    df_union = reduce(DataFrame.union, dfs)
    df_dedup = df_union.dropDuplicates(["id"])

    print(f"   Raw rows: {df_union.count()} | Unique (after dedup): {df_dedup.count()}")
    return df_dedup

print("    Kết nối S3 sẵn sàng")
print(f"   Mode: Wildcard path (hỗ trợ incremental subfolders)")

    Kết nối S3 sẵn sàng
   Mode: Wildcard path (hỗ trợ incremental subfolders)


In [0]:
# Test với bảng users trước
try:
    df_test = read_csv_from_s3("users")
    print(f"Đọc được! Số records: {df_test.count()}")
    df_test.show(3, truncate=False)
except Exception as e:
    print(f"Lỗi: {e}")

   [FALLBACK] Dùng file cũ: raw/users/users.csv
Đọc được! Số records: 218
+---+--------+---------+-------------------+
|id |role    |is_active|date_joined        |
+---+--------+---------+-------------------+
|6  |Admin   |true     |2026-05-22 05:30:47|
|10 |Customer|true     |2026-05-23 10:28:57|
|11 |Provider|true     |2026-05-23 10:30:14|
+---+--------+---------+-------------------+
only showing top 3 rows


In [0]:
import pyspark.sql.functions as F

tables = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']

results = {}

for t in tables:
    print(f"\n{'='*50}")
    print(f"ĐANG ĐỌC: {t.upper()}")
    
    try:
        df = read_csv_from_s3(t)
        count = df.count()
        
        # Đếm null từng cột
        nulls = {}
        for c in df.columns:
            n = df.filter(F.col(c).isNull()).count()
            if n > 0:
                nulls[c] = n
        
        results[t] = {
            "status": "---OK---",
            "count": count,
            "columns": list(df.columns),
            "nulls": nulls
        }
        
        print(f"---OK--- {count} records")
        print(f"   Columns: {list(df.columns)}")
        df.printSchema()
        df.show(3, truncate=False)
        
        if nulls:
            print(f"  Null ở cột: {nulls}")
        else:
            print("   Không có null")
            
    except Exception as e:
        results[t] = {"status": "---LỖI---", "error": str(e)}
        print(f"---LỖI--- khi đọc {t}: {e}")

print(f"\n{'='*50}")
print("ĐỌC XONG 6 BẢNG")


ĐANG ĐỌC: USERS
   [FALLBACK] Dùng file cũ: raw/users/users.csv
---OK--- 218 records
   Columns: ['id', 'role', 'is_active', 'date_joined']
root
 |-- id: long (nullable = true)
 |-- role: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- date_joined: string (nullable = true)

+---+--------+---------+-------------------+
|id |role    |is_active|date_joined        |
+---+--------+---------+-------------------+
|6  |Admin   |true     |2026-05-22 05:30:47|
|10 |Customer|true     |2026-05-23 10:28:57|
|11 |Provider|true     |2026-05-23 10:30:14|
+---+--------+---------+-------------------+
only showing top 3 rows
   Không có null

ĐANG ĐỌC: TOURS
   [FALLBACK] Dùng file cũ: raw/tours/tours.csv
---OK--- 772 records
   Columns: ['id', 'creator_id', 'title', 'price', 'departure_date', 'slots', 'status', 'created_at', 'category_names']
root
 |-- id: long (nullable = true)
 |-- creator_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price: double (null

In [0]:
# Schema mong đợi theo export_schema.md của [A]
EXPECTED_SCHEMAS = {
    "users":    ["id", "role", "is_active", "date_joined"],
    "tours":    ["id", "creator_id", "title", "price", "departure_date",
                 "slots", "status", "created_at", "category_names"],
    "bookings": ["id", "user_id", "tour_id", "number_of_people",
                 "total_price", "booking_date", "status", "created_at"],
    "payments": ["id", "booking_id", "amount", "payment_method",
                 "status", "created_at"],
    "revenues": ["id", "payment_id", "creator_id", "total_amount",
                 "creator_share", "admin_share", "created_at"],
    "reviews":  ["id", "user_id", "tour_id", "rating", "created_at"]
}

print("TỔNG KẾT VERIFY — NGÀY 1")
print("="*60)

schema_issues = []
encoding_issues = []
all_ok = True

for t, r in results.items():
    if r["status"] != "---OK---":
        print(f"  ---LỖI--- {t:12s} — KHÔNG ĐỌC ĐƯỢC: {r.get('error','')[:60]}")
        all_ok = False
        continue
    
    actual_cols   = [c.lower() for c in r["columns"]]
    expected_cols = EXPECTED_SCHEMAS.get(t, [])
    
    missing = [c for c in expected_cols if c not in actual_cols]
    extra   = [c for c in actual_cols   if c not in expected_cols]
    
    # Ghi nhận vấn đề schema
    if missing or extra:
        schema_issues.append({
            "table": t,
            "missing": missing,
            "extra": extra
        })
    
    warn = ""
    if missing: warn += f" |  ----THIẾU CỘT----: {missing}"
    if extra:   warn += f" |  -- cột thừa --: {extra}"
    if r["nulls"]: warn += f" | -- null --: {list(r['nulls'].keys())}"
    
    print(f" {t:12s} — {r['count']:5d} records{warn}")

print("="*60)

# Cảnh báo schema cho [D]
if schema_issues:
    print("\n  BÁO [D] (KHÁNH) — CÁC VẤN ĐỀ SCHEMA CẦN FIX:")
    for issue in schema_issues:
        t = issue["table"]
        if issue["missing"]:
            print(f"  ---LỖI--- Bảng [{t}] thiếu cột: {issue['missing']}")
            print(f"     -> Fix: bổ sung các cột này vào export_to_csv.py")
        if issue["extra"]:
            print(f"  ---LỖI---  Bảng [{t}] có cột thừa: {issue['extra']}")
            print(f"     -> Fix: bỏ các cột nhạy cảm này khỏi export_to_csv.py")
    all_ok = False

# Kiểm tra encoding tiếng Việt (bảng tours)
if "tours" in results and results["tours"]["status"] == "---OK---":
    print("\n Kiểm tra encoding tiếng Việt (bảng tours)...")
    df_tours = read_csv_from_s3("tours")
    
    # Kiểm tra cột title có tiếng Việt không bị lỗi
    if "title" in [c.lower() for c in df_tours.columns]:
        broken = df_tours.filter(
            F.col("title").contains("?") |
            F.col("title").contains("â€") |
            F.col("title").contains("Ã")
        ).count()
        
        if broken > 0:
            print(f"    {broken} dòng bị lỗi encoding tiếng Việt!")
            print(f"  -> Báo [D]: thêm encoding='utf-8-sig' vào export_to_csv.py")
            encoding_issues.append("tours.title")
        else:
            print("   Tiếng Việt hiển thị đúng")
    
    # Show thử title để kiểm tra bằng mắt
    print("\n  Kiểm tra bằng mắt — 3 tên tour:")
    df_tours.select("title").show(3, truncate=False)

# Kết luận cuối
print("\n" + "="*60)
if all_ok and not encoding_issues:
    print("KẾT LUẬN: Dữ liệu thật đã sẵn sàng!")
    print("Báo cả nhóm: có thể bắt đầu Ngày 2 (Bronze ingestion)")
else:
    print("KẾT LUẬN: Cần fix một số vấn đề trước khi sang Ngày 2")
    print("Gửi output này cho [D] (Khang) và [A] (Hà) để xử lý")
print("="*60)

TỔNG KẾT VERIFY — NGÀY 1
 users        —   218 records
 tours        —   772 records | -- null --: ['category_names']
 bookings     —  1321 records
 payments     —   794 records
 revenues     —   788 records
 reviews      —   271 records

 Kiểm tra encoding tiếng Việt (bảng tours)...
   [FALLBACK] Dùng file cũ: raw/tours/tours.csv
   Tiếng Việt hiển thị đúng

  Kiểm tra bằng mắt — 3 tên tour:
+--------------------------------------------------------------------+
|title                                                               |
+--------------------------------------------------------------------+
|Tour 3 đảo bằng Cano Nam Phú Quốc - 1 ngày                          |
|Private Luxury Tour | 3 đảo Nha Trang: Hòn Mun, Làng Chài, Bãi Tranh|
|Khu du lịch Sinh Thái Cần Giờ                                       |
+--------------------------------------------------------------------+
only showing top 3 rows

KẾT LUẬN: Dữ liệu thật đã sẵn sàng!
Báo cả nhóm: có thể bắt đầu Ngày 2 (Bronze ing